In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Set a clean style for research paper visualizations
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:

data_path = '../data/final_model_ready_pune_data_ver2.csv'

print(f"Loading finalized dataset from: {data_path}")
df = pd.read_csv(data_path)

# Convert date to datetime and sort chronologically
df['arrival_date'] = pd.to_datetime(df['arrival_date'])
df = df.sort_values(by='arrival_date').reset_index(drop=True)

# CHECKPOINT: Verify the data
print(f"Dataset Shape: {df.shape}")
print(f"Date Range: {df['arrival_date'].min().date()} to {df['arrival_date'].max().date()}")

# Display the first 3 rows to confirm features are intact
display(df.head(3))

Loading finalized dataset from: ../data/final_model_ready_pune_data_ver2.csv
Dataset Shape: (24734, 33)
Date Range: 2021-03-24 to 2026-02-21


,arrival_date,mandi_name,district,state,variety,min_price,max_price,modal_price,commodity,is_real_trade,...,sin_365_1,cos_365_1,sin_365_2,cos_365_2,target_price,temp_mean_lag1,rainfall_lag1,rainfall_7d_sum,rainfall_30d_sum,temp_7d_avg
0,2021-03-24,Manchar,Pune,Maharashtra,Other,250.0,1200.0,725.0,Onion,1,...,0.989794,0.142508,0.282108,-0.959383,725.0,26.7,1.6,1.7,2.3,27.185714
1,2021-03-25,Pune,Pune,Maharashtra,Local,500.0,1300.0,900.0,Onion,1,...,0.992099,0.125461,0.248940,-0.968519,900.0,27.6,0.0,1.7,1.7,27.128571
2,2021-03-25,Manchar,Pune,Maharashtra,Other,250.0,1200.0,725.0,NaN,0,...,0.992099,0.125461,0.248940,-0.968519,825.0,27.6,0.0,1.7,1.7,27.128571


In [3]:
# =========================================================
# STEP 2: Categorical Encoding and Chronological Split
# =========================================================

# 1. Prepare Categoricals
categorical_cols = ['mandi_name', 'district', 'state', 'variety']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')
        
print("Categorical features encoded.")

# 2. Chronological Split
# We train on past data and test on the most recent data
split_date = '2025-03-01' 
train_df = df[df['arrival_date'] < split_date].copy()
test_df = df[df['arrival_date'] >= split_date].copy()

# CHECKPOINT: Verify the split
print(f"Training data range: {train_df['arrival_date'].min().date()} to {train_df['arrival_date'].max().date()} (Rows: {len(train_df)})")
print(f"Testing data range: {test_df['arrival_date'].min().date()} to {test_df['arrival_date'].max().date()} (Rows: {len(test_df)})")

# 3. Separate features (X) and target (y)
drop_cols = ['arrival_date', 'target_price', 'commodity']

X_train = train_df.drop(columns=drop_cols, errors='ignore')
y_train = train_df['target_price']

X_test = test_df.drop(columns=drop_cols, errors='ignore')
y_test = test_df['target_price']

# 4. Create base LightGBM Datasets
# Setting free_raw_data=False allows us to reuse these datasets for multiple models
lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols, free_raw_data=False)
lgb_eval = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_cols, reference=lgb_train, free_raw_data=False)

print("\nTrain/Test split complete and LightGBM datasets created successfully!")

Categorical features encoded.
Training data range: 2021-03-24 to 2025-02-28 (Rows: 19651)
Testing data range: 2025-03-01 to 2026-02-21 (Rows: 5083)

Train/Test split complete and LightGBM datasets created successfully!


In [4]:
# =========================================================
# STEP 3: Train Probabilistic Models (Quantile Regression)
# =========================================================
import os

# Ensure the models directory exists relative to the notebooks folder
os.makedirs('../models', exist_ok=True)

# Base parameters shared across all models
base_params = {
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 45,
    'max_depth': 9,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

quantiles = {'p10': 0.10, 'p50': 0.50, 'p90': 0.90}
models = {}

print("Starting training for Probabilistic Models (1-Day Horizon)...\n")

for name, alpha in quantiles.items():
    print(f"--- Training {name} Model (Alpha={alpha}) ---")
    
    # Inject quantile objective and specific alpha
    params = base_params.copy()
    params['objective'] = 'quantile'
    params['alpha'] = alpha
    params['metric'] = 'quantile'

    # Callbacks to monitor training without cluttering the notebook output too much
    callbacks = [
        lgb.early_stopping(stopping_rounds=50, first_metric_only=False),
        lgb.log_evaluation(period=500)
    ]

    # Train the model
    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=2000,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=['train', 'eval'],
        callbacks=callbacks
    )
    
    models[name] = model
    
    # Save each model file to the production models directory
    model_path = f"../models/lightgbm_onion_1day_{name}.txt"
    model.save_model(model_path)
    print(f"Saved {name} model to: {model_path}\n")

print("CHECKPOINT: All 3 probabilistic models trained and saved successfully!")

Starting training for Probabilistic Models (1-Day Horizon)...

--- Training p10 Model (Alpha=0.1) ---
Training until validation scores don't improve for 50 rounds
[500]	train's quantile: 19.3086	eval's quantile: 27.3884
[1000]	train's quantile: 16.2086	eval's quantile: 21.254
Early stopping, best iteration is:
[1094]	train's quantile: 15.9676	eval's quantile: 21.2328
Saved p10 model to: ../models/lightgbm_onion_1day_p10.txt

--- Training p50 Model (Alpha=0.5) ---
Training until validation scores don't improve for 50 rounds
[500]	train's quantile: 35.7801	eval's quantile: 34.1408
[1000]	train's quantile: 33.4848	eval's quantile: 33.6833
Early stopping, best iteration is:
[994]	train's quantile: 33.4961	eval's quantile: 33.6781
Saved p50 model to: ../models/lightgbm_onion_1day_p50.txt

--- Training p90 Model (Alpha=0.9) ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[439]	train's quantile: 18.4713	eval's quantile: 21.576
Saved p90 mode

In [5]:
# =========================================================
# STEP 4: Inference and Probabilistic Metrics
# =========================================================
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import numpy as np

print("Generating predictions on unseen test data...")

# Initialize a dataframe to hold our actuals and predictions
results_df = test_df[['arrival_date', 'mandi_name', 'target_price']].copy()

# Generate predictions for each of the 3 quantile models
for name in quantiles.keys():
    results_df[f'{name}_pred'] = models[name].predict(X_test)

# 1. Standard Metrics on the Median (p50) Forecast
# In quantile regression, the p50 model acts as our standard "point prediction"
mae = mean_absolute_error(results_df['target_price'], results_df['p50_pred'])
mape = mean_absolute_percentage_error(results_df['target_price'], results_df['p50_pred'])
rmse = np.sqrt(mean_squared_error(results_df['target_price'], results_df['p50_pred']))

print("\n--- Standard Regression Metrics (on p50 Median Forecast) ---")
print(f"MAE:  ₹ {mae:.2f}")
print(f"RMSE: ₹ {rmse:.2f}")
print(f"MAPE: {mape*100:.2f}%")

# 2. Probabilistic Metric: Prediction Interval Coverage (PICP)
# For an 80% confidence interval (p10 to p90), a well-calibrated model 
# should capture the true price roughly 80% of the time.
results_df['in_bound'] = (results_df['target_price'] >= results_df['p10_pred']) & \
                         (results_df['target_price'] <= results_df['p90_pred'])
                         
coverage = results_df['in_bound'].mean() * 100

print("\n--- Probabilistic Calibration Metrics ---")
print(f"80% Prediction Interval Coverage: {coverage:.1f}%")
if coverage < 70:
    print("Note: The model is slightly overconfident (bands are too narrow).")
elif coverage > 90:
    print("Note: The model is underconfident (bands are too wide).")
else:
    print("Note: Excellent calibration! The uncertainty bands are highly reliable.")

Generating predictions on unseen test data...

--- Standard Regression Metrics (on p50 Median Forecast) ---
MAE:  ₹ 67.36
RMSE: ₹ 134.94
MAPE: 5.44%

--- Probabilistic Calibration Metrics ---
80% Prediction Interval Coverage: 71.5%
Note: Excellent calibration! The uncertainty bands are highly reliable.


In [8]:
# =========================================================
# REVISED STEP 3: Quantile-Specific Tuning
# =========================================================
import os
import lightgbm as lgb

os.makedirs('../models', exist_ok=True)

# Shared base parameters
base_params = {
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

# SPECIFIC tuning for each quantile to prevent extreme overfitting
quantile_configs = {
    'p50': {
        'alpha': 0.50,
        'max_depth': 9,          # Deeper trees for the median
        'num_leaves': 45,
        'min_data_in_leaf': 20,  # Standard minimum data
        'lambda_l1': 0.1,        # Low regularization
        'lambda_l2': 0.1
    },
    'p10': {
        'alpha': 0.10,
        'max_depth': 5,          # Shallower trees for extremes
        'num_leaves': 20,
        'min_data_in_leaf': 50,  # Require more data to make a split (smoothes it out)
        'lambda_l1': 1.5,        # High L1 Regularization to ignore noise
        'lambda_l2': 1.0
    },
    'p90': {
        'alpha': 0.90,
        'max_depth': 5,          # Shallower trees for extremes
        'num_leaves': 20,
        'min_data_in_leaf': 50,  # Require more data to make a split (smoothes it out)
        'lambda_l1': 1.5,        # High L1 Regularization to ignore noise
        'lambda_l2': 1.0
    }
}

models = {}
print("Starting Quantile-Specific Tuning Training...\n")

for name, config in quantile_configs.items():
    print(f"--- Training {name} Model ---")
    
    # Merge base params with specific config
    params = {**base_params, **config}
    params['objective'] = 'quantile'
    params['metric'] = 'quantile'

    callbacks = [
        lgb.early_stopping(stopping_rounds=50, first_metric_only=False),
        lgb.log_evaluation(period=500)
    ]

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=2000,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=['train', 'eval'],
        callbacks=callbacks
    )
    
    models[name] = model
    model_path = f"../models/lightgbm_onion_1day_{name}_ver2.txt"
    model.save_model(model_path)

print("\nCHECKPOINT: Tuned models trained and saved!")

Starting Quantile-Specific Tuning Training...

--- Training p50 Model ---
Training until validation scores don't improve for 50 rounds
[500]	train's quantile: 35.8375	eval's quantile: 34.1062
[1000]	train's quantile: 33.3944	eval's quantile: 33.5949
Early stopping, best iteration is:
[1120]	train's quantile: 33.052	eval's quantile: 33.5542
--- Training p10 Model ---
Training until validation scores don't improve for 50 rounds
[500]	train's quantile: 22.2815	eval's quantile: 26.4638
[1000]	train's quantile: 20.4082	eval's quantile: 21.2495
Early stopping, best iteration is:
[1384]	train's quantile: 19.7122	eval's quantile: 20.5753
--- Training p90 Model ---
Training until validation scores don't improve for 50 rounds
[500]	train's quantile: 22.0062	eval's quantile: 20.4661
[1000]	train's quantile: 20.1728	eval's quantile: 20.2005
Early stopping, best iteration is:
[955]	train's quantile: 20.3053	eval's quantile: 20.1689

CHECKPOINT: Tuned models trained and saved!


In [9]:
# =========================================================
# STEP 4: Inference and Probabilistic Metrics
# =========================================================
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import numpy as np

print("Generating predictions on unseen test data...")

# Initialize a dataframe to hold our actuals and predictions
results_df = test_df[['arrival_date', 'mandi_name', 'target_price']].copy()

# Generate predictions for each of the 3 quantile models
for name in quantiles.keys():
    results_df[f'{name}_pred'] = models[name].predict(X_test)

# 1. Standard Metrics on the Median (p50) Forecast
# In quantile regression, the p50 model acts as our standard "point prediction"
mae = mean_absolute_error(results_df['target_price'], results_df['p50_pred'])
mape = mean_absolute_percentage_error(results_df['target_price'], results_df['p50_pred'])
rmse = np.sqrt(mean_squared_error(results_df['target_price'], results_df['p50_pred']))

print("\n--- Standard Regression Metrics (on p50 Median Forecast) ---")
print(f"MAE:  ₹ {mae:.2f}")
print(f"RMSE: ₹ {rmse:.2f}")
print(f"MAPE: {mape*100:.2f}%")

# 2. Probabilistic Metric: Prediction Interval Coverage (PICP)
# For an 80% confidence interval (p10 to p90), a well-calibrated model 
# should capture the true price roughly 80% of the time.
results_df['in_bound'] = (results_df['target_price'] >= results_df['p10_pred']) & \
                         (results_df['target_price'] <= results_df['p90_pred'])
                         
coverage = results_df['in_bound'].mean() * 100

print("\n--- Probabilistic Calibration Metrics ---")
print(f"80% Prediction Interval Coverage: {coverage:.1f}%")
if coverage < 70:
    print("Note: The model is slightly overconfident (bands are too narrow).")
elif coverage > 90:
    print("Note: The model is underconfident (bands are too wide).")
else:
    print("Note: Excellent calibration! The uncertainty bands are highly reliable.")

Generating predictions on unseen test data...

--- Standard Regression Metrics (on p50 Median Forecast) ---
MAE:  ₹ 67.11
RMSE: ₹ 134.93
MAPE: 5.43%

--- Probabilistic Calibration Metrics ---
80% Prediction Interval Coverage: 78.6%
Note: Excellent calibration! The uncertainty bands are highly reliable.
